In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
import re
from wordcloud import WordCloud
from rouge_score import rouge_scorer
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Please uncomment if you want to download
# nltk.download('punkt')
# nltk.download('punkt_tab')      
# nltk.download('wordnet')    
# nltk.download('omw-1.4') 
# nltk.download('averaged_perceptron_tagger_eng') 

In [ ]:
try:
    df = pd.read_csv('filtered_data.csv', nrows=10000)
    df.to_csv('10000only.csv', index=False)
except:
    df = pd.read_csv('10000only.csv')
finally:
    display(df)


# Data Exploration

In [ ]:
df['article_len'] = df['article'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10,6))
plt.hist(df['article_len'], bins=50)
plt.title("Article Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df['summary_len'] = df['highlights'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10,6))
plt.hist(df['summary_len'], bins=50)
plt.title("Summary Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df['article_len'], df['summary_len'],s=10)
plt.title("Article Length vs Summary Length")
plt.xlabel("Article Length")
plt.ylabel("Summary Length")
plt.show()

In [ ]:
# simple cleaning for visualization
def clean_text(text):
    words = text.lower().split()
    words = [w for w in words if w.isalpha()]
    return " ".join(words)

articles_text = " ".join(df['article'].apply(clean_text))
highlights_text = " ".join(df['highlights'].apply(clean_text))

wordcloud_articles = WordCloud(
    width=800,
    height=400,
    background_color='white'
).generate(articles_text)

wordcloud_highlights = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='viridis'
).generate(highlights_text)


fig, ax = plt.subplots(1, 2, figsize=(15,6))

ax[0].imshow(wordcloud_articles, interpolation='bilinear')
ax[0].set_title("Articles")
ax[0].axis("off")

ax[1].imshow(wordcloud_highlights, interpolation='bilinear')
ax[1].set_title("Highlights")
ax[1].axis("off")

plt.show()

# Preprocessing

In [ ]:
df_clean = pd.DataFrame()
df_clean = df.drop(columns=['id','article_len','summary_len']).copy()

In [ ]:
df_clean

In [ ]:
# remove bracketed publisher info
pattern1 = re.compile(
    r'^(?:[a-z,]+\s+){0,3}\((?:[a-z\.]+\s*){1,2}\)(?:\s+--\s+)?',
    re.IGNORECASE
)

# remove bylines, social media follows , publication dates, and update timestamps
pattern2 = re.compile(
    # r'^(?:By\s+.*?[a-z,\.@ ]+?\s+\.\s+)?(?:follow.*?\s*\.\s*)?(?:PUBLISHED:.*?\|)?(?:.*?UPDATED:\s\..*?\d{1,2}:\d{2}\s+[a-z\.]{2,5},\s*\d{1,2}\s+\w+\s+\d{4}\s*\.\s*)?',
    # r'^(?:By\s+.*?(?:[\w,\.@]+?\s+\.\s+){0,3})?(?:PUBLISHED:.*?\|)?(?:.*?UPDATED:\s\..*?\d{1,2}:\d{2}\s+[a-z\.]{2,5},\s*\d{1,2}\s+\w+\s+\d{4}\s*\.\s*)?',
    r'^(?:By\s+.*?(?:(?:[\w,\.@\s\[\]\']+){0,3}\s+\.\s+){0,3})?(?:PUBLISHED:.*?\|)?(?:.*?UPDATED:\s\..*?\d{1,2}:\d{2}\s+[a-z\.]{2,5},\s*\d{1,2}\s+\w+\s+\d{4}\s+\.\s+)?',
    re.IGNORECASE
)

# remove "Last updated" lines
pattern3 = re.compile(
    r'^Last updated.*?\s*\.\s*',
    re.IGNORECASE
)

In [ ]:
lemmatizer= WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

### Function Definitions

In [ ]:
def remove_prefix(text):
    text = pattern1.sub('', text)
    text = pattern2.sub('', text)
    text = pattern3.sub('', text)
    return text

In [ ]:
# remove new lines \n
def organize_highlights(text):
    sentences = sent_tokenize(text)
    return " ".join(sentences)

### Apply functions on dataframe

In [ ]:
df_clean["article"] = df_clean["article"].apply(remove_prefix)
df_clean["highlights"] = df_clean["highlights"].apply(organize_highlights)
df_clean

In [ ]:
def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    return tokens

# regex debug

In [ ]:
pattern1.findall("(EW.com) -- ")

In [ ]:
pattern2.findall(df["article"][145])

In [ ]:
df_clean["article"][0]

In [ ]:
df_clean["article"][9]

In [ ]:
df_clean["article"][20]

In [ ]:
df_clean["article"][31]

In [ ]:
df_clean["article"][37]

In [ ]:
df_clean["article"][83]

In [ ]:
df_clean["article"][100]

In [ ]:
df_clean["article"][119]

In [ ]:
df_clean["article"][145]

In [ ]:
df_clean["article"][166]

In [ ]:
df_clean["article"][188]

In [ ]:
df_clean["article"][271]

In [ ]:
df_clean["article"][374]

In [ ]:
df_clean["article"][488]

In [ ]:
df_clean["article"][9997]

In [ ]:
df_clean.to_csv('cleaned_data.csv', index=False)

# Load cleaned data directly to save time

In [ ]:
df_clean = pd.read_csv('cleaned_data.csv')

# Logistic Regression

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True) # use stemmer on top of lemmatization for better matching    

In [ ]:
def prepare_data(df, threshold=0.15):
    X = []
    y = []

    for _, row in df.iterrows():

        article_sentences = sent_tokenize(row['article'].lower())
        highlights_tokens =  preprocess(row['highlights'])

        highlights_text = " ".join(highlights_tokens)

        for sentence in article_sentences:

            words = preprocess(sentence)
            sentence_text = " ".join(words)

            # ROUGE similarity score
            score = scorer.score(sentence_text, highlights_text)['rouge1'].fmeasure

            label = 1 if score > threshold else 0

            X.append(sentence_text)
            y.append(label)

    return X, y

In [ ]:
sentences, labels = prepare_data(df_clean)
vectorizer = TfidfVectorizer()
X_vectors = vectorizer.fit_transform(sentences)

In [ ]:
X_vectors.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_vectors, labels, test_size=0.2, random_state=42)

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

# ML Evaluation

In [ ]:
preds = lr.predict(X_test)
df_ml_comp = pd.DataFrame({'Actual': y_test[:10], 'Predicted': preds[:10]})
df_ml_comp

In [ ]:
print("Logistic Regression Accuracy:", accuracy_score(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))

In [ ]:
f1_score(y_test, preds)

In [ ]:
pd.Series(labels).value_counts()

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x=labels)
plt.title("Label Distribution")
plt.xlabel("Label")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Accuracys is not a good metric for this task due to class imbalance and the nature of summarization.

### Baseline Model

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)
preds_baseline = baseline.predict(X_test)

In [ ]:
print("Baseline Accuracy:", accuracy_score(y_test, preds_baseline))
print("\nClassification Report:\n", classification_report(y_test, preds_baseline, zero_division=0))

### Tuned Model

In [ ]:
logreg = LogisticRegression(max_iter=1000)

# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10],                     # Regularization strength, smaller values specify stronger regularization (may underfit). vice versa for larger values (may overfit)
    'penalty': ['l2'],                     # Penalty type, l2 is standard and more stable with sparse data.
    'solver': ['lbfgs', 'liblinear'],      # solvers that support l2
    'class_weight': [None, 'balanced']     # important for imbalanced data
}

# Grid search
grid = GridSearchCV(
    estimator=logreg,
    param_grid=param_grid,
    scoring='f1',     # accuracy is not suitable
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Train
grid.fit(X_train, y_train)

# Best results
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

In [ ]:
lr_best = grid.best_estimator_
preds_best = lr_best.predict(X_test)

In [ ]:
cm_baseline = confusion_matrix(y_test, preds_baseline)
cm_lr = confusion_matrix(y_test, preds)
cm_lr_best = confusion_matrix(y_test, preds_best)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title("Baseline")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title("LogReg")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

sns.heatmap(cm_lr_best, annot=True, fmt='d', cmap='Blues', ax=axes[2])
axes[2].set_title("LogReg Best")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
def summarize_article(article, model, vectorizer, num_sentences=3):

    source_sentences = sent_tokenize(article)

    pre_sentences = [" ".join(preprocess(sentence)) for sentence in source_sentences]
    sentence_vectors = vectorizer.transform(pre_sentences)

    scores = model.predict_proba(sentence_vectors)[:, 1]

    ranked_indices = np.argsort(scores)[::-1]
    
    # Select top-k sentences
    top_indices = sorted(ranked_indices[:num_sentences])

    summary = " ".join([source_sentences[i] for i in top_indices])

    return summary

In [ ]:
def get_rouge_score(model, vectorizer, num_sentences=3):
    scores = []
    for row in df_clean.itertuples():
        article = row.article
        highlight = row.highlights

        generated_summary = summarize_article(article, model, vectorizer, num_sentences)

        score = scorer.score(generated_summary, highlight)['rouge1'].fmeasure
        scores.append(score)
        
    return scores

In [ ]:
def find_rouge_outliers(scores):
    scores = np.array(scores)

    q1 = np.percentile(scores, 25)
    q3 = np.percentile(scores, 75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_indices = np.where((scores < lower_bound) | (scores > upper_bound))[0]

    return {
        "outlier_indices": outlier_indices,
        "outlier_scores": scores[outlier_indices],
        "bounds": (lower_bound, upper_bound)
    }

In [ ]:
rouge_scores = get_rouge_score(lr, vectorizer)
len(rouge_scores)

In [ ]:
np.median(rouge_scores)

In [ ]:
find_rouge_outliers(rouge_scores)

In [ ]:
# Since outliers are present use median for ROUGE score representation
df_metrics = pd.DataFrame({
    "Baseline": [
        accuracy_score(y_test, preds_baseline),
        precision_score(y_test, preds_baseline, zero_division=0),
        recall_score(y_test, preds_baseline, zero_division=0),
        f1_score(y_test, preds_baseline, zero_division=0),
        np.median(get_rouge_score(baseline, vectorizer))
    ],
    "LogReg": [
        accuracy_score(y_test, preds),
        precision_score(y_test, preds, zero_division=0),
        recall_score(y_test, preds, zero_division=0),
        f1_score(y_test, preds, zero_division=0),
        np.median(get_rouge_score(lr, vectorizer))
    ],
    "LogReg Best": [
        accuracy_score(y_test, preds_best),
        precision_score(y_test, preds_best, zero_division=0),
        recall_score(y_test, preds_best, zero_division=0),
        f1_score(y_test, preds_best, zero_division=0),
        np.median(get_rouge_score(lr_best, vectorizer))
    ]
}, index=["Accuracy", "Precision", "Recall", "F1", "ROUGE-1 F1"])

display(df_metrics)

# Try getting extractive summaries (uncomment inputs)

In [ ]:
article = 0
summary_sents = 3
# article = int(input("Enter the article index (0-9999): "))
# summary_sents = int(input("Enter the number of sentences for the summary: "))
test_article = df_clean['article'].iloc[article]
actual_highlight = df_clean['highlights'].iloc[article]

generated_summary = summarize_article(test_article, lr_best, vectorizer, summary_sents)

print("\n------- Original Article Snippet ---")
print(test_article[:300] + "...")
print("\n--- Actual Highlight -------")
print(actual_highlight)
print("\n--- Logistic Regression Predicted Summary ---")
print(generated_summary)